# GM6208-150T rotor inertia — alignment snap fit

Extracts `MotorParams.j_kg_m2` from the alignment event's step response, then
converts the coast-down ratios into absolute `b_nm_per_rad_s` and the Coulomb
torque.

**Input:** `alignment_snap.csv` — `PCS_BENCH_DUTY_SEQ = 2` telemetry
(`motor_angle` + vsense at 2 ms) spanning one button tap: the firmware holds
the U+/V− vector at 0.1 duty for 500 ms and the rotor snaps to it.

**Method:** every constant except J is already measured — `Ke` and the shape
(sinusoidal) from the spin capture, `R_LL` and `vbus` from the stall capture,
`B/J` and `T_c/J` from the coasts. The snap obeys
`J·θ̈ = −√3·Ke·I·sin(p·(θ−θeq)) − B·θ̇ − T_c·sign(θ̇)` with
`I = 0.1·vbus/R_LL`; a grid search over J simulates the ODE and least-squares
it against the measured angle. Run with the repo venv kernel (`.venv`).

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CSV = Path("alignment_snap.csv")

# Measured constants from the companion captures (stall + free-spin).
KE_V_PER_RAD = 0.55        # per-phase, sinusoidal shape (bemf_spin)
R_LL_OHM = 29.35           # terminal resistance (stall_current_measure_r)
VBUS_V = 19.57             # stall capture's bus reading
ALIGN_DUTY = 0.1           # ALIGNMENT_DUTY_CYCLE in app_motorControl.c
POLE_PAIRS = 14
B_OVER_J = 2.258           # 1/s      (bemf_spin coast fit)
TC_OVER_J = 13.864         # rad/s^2  (bemf_spin coast fit)

I_ALIGN_A = ALIGN_DUTY * VBUS_V / R_LL_OHM
TE_PK_NM_PER_J = None      # set below once J is scanned
TE_PK_NM = np.sqrt(3.0) * KE_V_PER_RAD * I_ALIGN_A
K_SPRING = TE_PK_NM * POLE_PAIRS
print(f"alignment current  : {I_ALIGN_A * 1e3:.1f} mA")
print(f"peak align torque  : {TE_PK_NM * 1e3:.2f} mNm")
print(f"spring stiffness   : {K_SPRING:.3f} Nm/rad (at equilibrium)")

In [ ]:
raw = defaultdict(list)
with open(CSV) as f:
    for row in csv.DictReader(f):
        raw[row["signalName"].split("/")[-1]].append((int(row["t"]), float(row["value"])))
t_ms = np.array([t for t, _ in raw["motor_angle"]])
ang = np.array([x for _, x in raw["motor_angle"]])
t = (t_ms - t_ms[0]) * 1e-3
theta = np.unwrap(np.deg2rad(ang))
omega = np.gradient(theta, t)

# The snap: first sustained motion. Fit window runs from just before motion to
# well after settling.
moving = np.abs(np.convolve(omega, np.ones(5) / 5, mode="same")) > 0.5
i0 = int(np.flatnonzero(moving)[0]) - 5
i1 = int(np.flatnonzero(moving)[-1]) + 100
tw = t[i0:i1] - t[i0]
thw = theta[i0:i1]
theta_eq = float(np.mean(thw[-50:]))
print(f"snap window: {t[i0]:.2f}..{t[i1]:.2f} s  ({i1 - i0} samples)")
print(f"swing: {np.rad2deg(abs(theta_eq - thw[0])):.2f} deg mech "
      f"({np.rad2deg(abs(theta_eq - thw[0])) * POLE_PAIRS:.0f} deg elec)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tw * 1e3, np.rad2deg(thw), ".", ms=3)
ax.axhline(np.rad2deg(theta_eq), color="0.6", lw=0.8)
ax.set_xlabel("t from snap [ms]")
ax.set_ylabel("motor angle [deg]")
ax.set_title("alignment snap")
fig.tight_layout()

## ODE fit over J

The snap is a *driven-circuit* event: the rotor's motion generates loop BEMF
that modulates the alignment current, adding electrical damping
`(√3·Ke·sinφ)²/R_LL` — an order above the mechanical friction away from
equilibrium (and zero at it). For each candidate J the simulator integrates
the quasi-static loop (`I = (0.1·vbus − e_loop)/R_LL`, τ_elec ≪ the
mechanics) with `B` and `T_c` pinned by the coast ratios, from the measured
rest angle. The reported J minimizes the rms angle error; the error valley's
width gives the confidence band.

In [ ]:
def simulate(j, t_grid, th0):
    b = B_OVER_J * j
    tc = TC_OVER_J * j
    v_drive = ALIGN_DUTY * VBUS_V
    dt = 1e-4
    th, w = th0, 0.0
    out = np.empty_like(t_grid)
    tt = 0.0
    k = 0
    while k < len(t_grid):
        while tt < t_grid[k]:
            phi = POLE_PAIRS * (th - theta_eq)
            lam_p = -np.sqrt(3.0) * KE_V_PER_RAD * np.sin(phi)  # dλ/dθ_mech
            i_loop = (v_drive - lam_p * w) / R_LL_OHM           # quasi-static
            te = lam_p * i_loop
            fric = b * w + (tc * np.sign(w) if abs(w) > 1e-6 else
                            np.clip(te, -tc, tc))
            w += (te - fric) * dt / j
            th += w * dt
            tt += dt
        out[k] = th
        k += 1
    return out

js = np.geomspace(2e-5, 2e-3, 60)
errs = np.array([np.sqrt(np.mean((simulate(j, tw, thw[0]) - thw) ** 2)) for j in js])
j_best = float(js[int(np.argmin(errs))])
# refine around the winner
js2 = np.linspace(j_best / 1.6, j_best * 1.6, 60)
errs2 = np.array([np.sqrt(np.mean((simulate(j, tw, thw[0]) - thw) ** 2)) for j in js2])
j_fit = float(js2[int(np.argmin(errs2))])
e_min = float(errs2.min())
band = js2[errs2 < 1.5 * e_min]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogx(js, np.rad2deg(errs), ".-", ms=3)
axes[0].semilogx(js2, np.rad2deg(errs2), ".-", ms=3, color="tab:orange")
axes[0].axvline(j_fit, color="r", lw=0.8)
axes[0].set_xlabel("J [kg·m²]")
axes[0].set_ylabel("rms angle error [deg]")
axes[0].set_title("fit error valley")
axes[1].plot(tw * 1e3, np.rad2deg(thw), ".", ms=3, label="measured")
axes[1].plot(tw * 1e3, np.rad2deg(simulate(j_fit, tw, thw[0])), "r-", lw=1.2,
             label=f"model, J = {j_fit:.2e}")
axes[1].set_xlabel("t from snap [ms]")
axes[1].set_ylabel("motor angle [deg]")
axes[1].legend()
axes[1].set_title("snap vs fitted model")
fig.tight_layout()

b_elec_pk = 3.0 * KE_V_PER_RAD ** 2 / R_LL_OHM
print("=" * 62)
print("GM6208-150T mechanical parameters — alignment snap fit")
print("=" * 62)
print(f"J (inertia)          : {j_fit:.3e} kg·m²  "
      f"(1.5x-rms band {band.min():.2e}..{band.max():.2e})")
print(f"rms fit error        : {np.rad2deg(e_min):.3f} deg")
print(f"B  = (B/J)·J         : {B_OVER_J * j_fit:.3e} Nm·s/rad   <- b_nm_per_rad_s")
print(f"Tc = (Tc/J)·J        : {TC_OVER_J * j_fit:.3e} Nm        (Coulomb, future term)")
print(f"electrical damping   : up to {b_elec_pk:.3e} Nm·s/rad (loop, away from eq)")
print(f"omega_n = sqrt(K/J)  : {np.sqrt(K_SPRING / j_fit):.1f} rad/s")
print("=" * 62)

## Caveats

- The stiction clip in the simulator (static friction holds until the spring
  exceeds `T_c`) also sets where the rotor parks: within `±T_c/K` of true
  equilibrium, so the settled angle is a fair `theta_eq` estimate.
- `B/J` and `T_c/J` ride in from the coast fit; errors there scale the fitted
  J weakly (the snap is spring-dominated: `K ≫ B·ω_n`).
- The encoder's noise floor (~0.03°) is far below the fit residual scale; the
  2 ms cadence gives ~40 points across the rise, which the error valley shows
  is enough to pin J to tens of percent — ballpark grade, as intended.